# Model 01 — Portfolio Risk Surface

This research notebook builds a transparent analytical model for how a portfolio's stressed potential loss changes with annualized volatility, position size, and stop-loss distance. It uses illustrative assumptions only; no market data is fabricated or downloaded.

## Research question

For a fixed portfolio value, how do exposure size and the stop-loss rule interact with market volatility to determine a conservative one-period loss estimate?

## Mathematical assumptions

We model one linear position with portfolio value $V$ and position size $w$ as a fraction of portfolio value. Annualized volatility is $\sigma$, the stop-loss distance is $d$, and the holding horizon is $h$ trading days. Under a normal-return approximation, the one-sided adverse volatility shock at confidence level $\alpha$ is:

$$q_{\alpha} = z_{\alpha}\sigma\sqrt{\frac{h}{D}}$$

where $D$ is the number of trading days per year and $z_{\alpha}$ is the standard-normal quantile. The modeled loss rate is the planned stop distance plus this adverse shock:

$$r_{loss} = d + q_{\alpha}$$

Therefore, modeled potential loss is:

$$L(V,w,\sigma,d) = Vw\left(d + z_{\alpha}\sigma\sqrt{\frac{h}{D}}\right)$$

This additive form is a conservative scenario convention: the stop-loss term represents the operational loss threshold and the volatility term represents additional adverse movement over the horizon.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models" / "risk").exists():
    PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models" / "risk" / "01_risk_surface"
sys.path.insert(0, str(MODEL_DIR))

from portfolio_risk_surface import modeled_loss, sensitivity_table

plt.style.use("seaborn-v0_8-whitegrid")

## Illustrative scenario inputs

These values are assumptions for demonstrating the model, not observations of any market or security.

In [ ]:
PORTFOLIO_VALUE = 100_000
CONFIDENCE = 0.99
HORIZON_DAYS = 1
TRADING_DAYS = 252

volatility_grid = np.linspace(0.10, 0.60, 60)
position_grid = np.linspace(0.05, 1.00, 60)
stop_loss_distance = 0.05

print(f"Portfolio value: ${PORTFOLIO_VALUE:,.0f}")
print(f"Confidence: {CONFIDENCE:.0%}; horizon: {HORIZON_DAYS} trading day")
print(f"Fixed stop-loss distance for surface: {stop_loss_distance:.1%}")

In [ ]:
risk_surface = modeled_loss(
    volatility_grid[:, None],
    position_grid[None, :],
    stop_loss_distance,
    portfolio_value=PORTFOLIO_VALUE,
    confidence=CONFIDENCE,
    horizon_days=HORIZON_DAYS,
    trading_days=TRADING_DAYS,
)

volatility_mesh, position_mesh = np.meshgrid(volatility_grid, position_grid, indexing="ij")
risk_surface.shape

## 3D portfolio risk surface

The x-axis is annualized volatility, the y-axis is position size, and the z-axis is modeled potential loss. With the stop-loss distance held constant, the surface is linear in both displayed inputs; the slope in volatility is scaled by position size.

In [ ]:
figure = plt.figure(figsize=(11, 8))
axis = figure.add_subplot(111, projection="3d")
surface = axis.plot_surface(
    volatility_mesh * 100,
    position_mesh * 100,
    risk_surface,
    cmap="viridis",
    edgecolor="none",
    alpha=0.9,
)
axis.set_xlabel("Annualized volatility (%)", labelpad=10)
axis.set_ylabel("Position size (% of portfolio)", labelpad=10)
axis.set_zlabel("Modeled potential loss ($)", labelpad=10)
axis.set_title("Portfolio Risk Surface — 99% One-Day Stress Scenario")
figure.colorbar(surface, ax=axis, shrink=0.6, pad=0.1, label="Modeled loss ($)")
plt.show()

## Sensitivity analysis

Each table varies one input while holding the other assumptions at a representative baseline. This isolates the model's marginal behavior.

In [ ]:
baseline = dict(
    annualized_volatility=0.30,
    position_size=0.25,
    stop_loss_distance=0.05,
    portfolio_value=PORTFOLIO_VALUE,
    confidence=CONFIDENCE,
    horizon_days=HORIZON_DAYS,
    trading_days=TRADING_DAYS,
)

volatility_sensitivity = sensitivity_table(
    "annualized_volatility", np.array([0.10, 0.20, 0.30, 0.40, 0.50]), **{k: v for k, v in baseline.items() if k != "annualized_volatility"}
)
position_sensitivity = sensitivity_table(
    "position_size", np.array([0.10, 0.25, 0.50, 0.75, 1.00]), **{k: v for k, v in baseline.items() if k != "position_size"}
)
stop_sensitivity = sensitivity_table(
    "stop_loss_distance", np.array([0.01, 0.03, 0.05, 0.07, 0.10]), **{k: v for k, v in baseline.items() if k != "stop_loss_distance"}
)

display(volatility_sensitivity.style.format({"annualized_volatility": "{:.0%}", "modeled_loss": "${:,.0f}"}))
display(position_sensitivity.style.format({"position_size": "{:.0%}", "modeled_loss": "${:,.0f}"}))
display(stop_sensitivity.style.format({"stop_loss_distance": "{:.0%}", "modeled_loss": "${:,.0f}"}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, table, x_column, label in [
    (axes[0], volatility_sensitivity, "annualized_volatility", "Annualized volatility (%)"),
    (axes[1], position_sensitivity, "position_size", "Position size (%)"),
    (axes[2], stop_sensitivity, "stop_loss_distance", "Stop-loss distance (%)"),
]:
    axis.plot(table[x_column] * 100, table["modeled_loss"], marker="o")
    axis.set_xlabel(label)
    axis.set_ylabel("Modeled loss ($)")
    axis.yaxis.set_major_formatter(lambda value, _: f"${value:,.0f}")
    axis.set_title(f"Loss vs {label.split(' (')[0]}")
fig.suptitle("One-at-a-time sensitivity around the baseline scenario")
fig.tight_layout()
plt.show()

## Interpretation

The surface rises as either annualized volatility or position size increases. Position size scales the entire loss estimate, so doubling exposure doubles both the stop-loss component and the volatility-stress component. At a fixed position size, higher volatility increases the modeled loss in proportion to the one-day volatility shock. Increasing the stop-loss distance shifts the entire surface upward because it adds directly to the loss rate.

The interaction is economically important: volatility sensitivity is larger for larger positions. A risk manager can therefore reduce the surface most directly by reducing exposure, tightening the stop distance, or avoiding positions whose volatility makes the stress component dominant.

## Limitations and assumptions

- Returns are approximated by a normal distribution and summarized by a constant annualized volatility.
- The model is for one linear position; it does not model diversification, correlations, options convexity, or portfolio netting.
- The stop is assumed to execute at the intended level. Gaps, slippage, fees, financing, taxes, and liquidity are excluded.
- The additive stop-plus-shock convention is conservative but is not a statistical maximum-loss guarantee.
- Volatility is an illustrative input, not an estimate fitted to observed market data.
- A one-day normal quantile does not capture fat tails, skew, volatility clustering, or regime changes.

A production extension should compare this analytical surface with historical, filtered-historical, and Monte Carlo scenarios; model correlations and nonlinear instruments; and include execution and liquidity assumptions.